# Project 2: Do Transformers Actually Beat a Linear Model at Forecasting?
**Naive / Seasonal-naive vs DLinear vs LSTM vs small Transformer**

Reference: Zeng et al., *Are Transformers Effective for Time Series Forecasting?* (AAAI 2023).

**Before you start:** Runtime > Change runtime type > **T4 GPU** (CPU works but is slower). Run cells top to bottom.

Two data options (set in the Config cell):
- `DATA_SOURCE = "ett"` : the ETTh1 benchmark (hourly electricity-transformer oil temperature). Downloads automatically, so it is the safe choice.
- `DATA_SOURCE = "csv"` : your own CSV, for example Bengaluru air quality (PM2.5). Upload it to Colab and fill in the column names.

In [ ]:
import os, time, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---------------- Config (edit here) ----------------
DATA_SOURCE = "ett"        # "ett" or "csv"

# --- used only if DATA_SOURCE == "csv" ---
CSV_PATH   = "city_hour.csv"   # upload to Colab (left sidebar > Files)
DATE_COL   = "Datetime"
TARGET_COL = "PM2.5"
FILTER_COL, FILTER_VAL = "City", "Bengaluru"   # set both to None if your file is already one city
FREQ   = "h"               # "h" hourly, "D" daily
SEASON = 24                # seasonal period in steps: 24 for hourly (daily cycle), 7 for daily data
CITY = dict(path=CSV_PATH, date=DATE_COL, target=TARGET_COL, filter_col=FILTER_COL, filter_val=FILTER_VAL,
            freq=FREQ, season=SEASON)   # snapshot, so the city cells keep working when DATA_SOURCE = "ett"

# --- experiment settings ---
LOOKBACK = 168             # input window (steps). 168 hours = 1 week
HORIZONS = [24, 168]       # forecast lengths to compare
SEEDS = [0, 1, 2]          # repeat each trained model 3 times -> mean +/- std
MAX_EPOCHS, PATIENCE, BATCH, LR = 30, 5, 64, 1e-3

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## 0. Upload the city CSV (Bengaluru air quality)
Run this once. It asks you to pick the CSV (e.g. Kaggle *Air Quality Data in India* -> `city_hour.csv`) and is skipped if the file is already in the Colab Files sidebar. To forecast on this data instead of ETTh1, set `DATA_SOURCE = "csv"` in the Config cell.

In [ ]:
if os.path.exists(CITY["path"]):
    print("Found", CITY["path"], "- nothing to upload.")
else:
    try:
        from google.colab import files
        uploaded = files.upload()                 # pick your CSV in the dialog
        if uploaded:
            name = next(iter(uploaded))
            CITY["path"] = name
            CSV_PATH = name                       # keeps DATA_SOURCE = "csv" pointing at the same file
            print("Uploaded:", name)
        else:
            print("No file chosen - the city cells will be skipped.")
    except ImportError:
        print(f"Not running in Colab. Put '{CITY['path']}' next to the notebook instead.")

## 1. Load the data

In [ ]:
if DATA_SOURCE == "ett":
    url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
    df = pd.read_csv(url).iloc[:14400]            # standard 12/4/4-month benchmark span
    DATE_COL, TARGET_COL, FREQ, SEASON = "date", "OT", "h", 24
    TRAIN_FRAC, VAL_FRAC = 0.6, 0.2
else:
    df = pd.read_csv(CSV_PATH)
    if FILTER_COL:
        df = df[df[FILTER_COL] == FILTER_VAL]
    TRAIN_FRAC, VAL_FRAC = 0.7, 0.1

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
s = (df.sort_values(DATE_COL).set_index(DATE_COL)[TARGET_COL].astype(float)
       .resample(FREQ).mean())
print("missing values before filling:", int(s.isna().sum()), "of", len(s))
s = s.interpolate(limit_direction="both")          # fill gaps by linear interpolation
series = s.values
dates = s.index
print("length:", len(series), "| from", dates[0], "to", dates[-1])

In [ ]:
n = len(series)
n_tr, n_va = int(n * TRAIN_FRAC), int(n * VAL_FRAC)
mu, sd = series[:n_tr].mean(), series[:n_tr].std()    # statistics from TRAIN only (no leakage)
z = (series - mu) / sd

fig, ax = plt.subplots(2, 1, figsize=(12, 6))
ax[0].plot(dates, series, lw=0.6)
ax[0].axvline(dates[n_tr], color="r", ls="--"); ax[0].axvline(dates[n_tr + n_va], color="r", ls="--")
ax[0].set_title(f"{TARGET_COL}: train | val | test (chronological split)")
ax[1].plot(dates[:SEASON * 14], series[:SEASON * 14])
ax[1].set_title("First 2 weeks (look for the daily cycle)")
plt.tight_layout(); plt.savefig("data_overview.png", dpi=150); plt.show()
print(f"train {n_tr} | val {n_va} | test {n - n_tr - n_va}")

## 1b. Visualise the city data (Bengaluru PM2.5)
Works whichever `DATA_SOURCE` is selected. Gaps are left as gaps here (not interpolated) so you can see how much of the series is actually missing. The daily cycle and the autocorrelation peaks at multiples of 24 are the evidence for `SEASON = 24` and the 168-step look-back.

In [ ]:
from matplotlib.gridspec import GridSpec

if not os.path.exists(CITY["path"]):
    print(f"'{CITY['path']}' not found. Run the upload cell (section 0) first, then re-run this cell.")
else:
    d = pd.read_csv(CITY["path"])
    if CITY["filter_col"]:
        d = d[d[CITY["filter_col"]] == CITY["filter_val"]]
    assert len(d), (f"No rows where {CITY['filter_col']} == {CITY['filter_val']!r}. "
                    f"Values present: {sorted(pd.read_csv(CITY['path'])[CITY['filter_col']].unique())[:30]}")
    d[CITY["date"]] = pd.to_datetime(d[CITY["date"]])
    pm = (d.sort_values(CITY["date"]).set_index(CITY["date"])[CITY["target"]]
            .astype(float).resample(CITY["freq"]).mean())          # NaN gaps are kept on purpose
    miss = pm.isna()
    hourly = CITY["freq"].lower() == "h"
    label = CITY["filter_val"] or "city"
    vals = pm.dropna()

    run_id = (miss != miss.shift()).cumsum()
    gap_len = miss.groupby(run_id).transform("sum")                  # length of the missing run each step belongs to
    longest_gap = int(gap_len.max())
    long_gap = miss & (gap_len >= (24 if hourly else 3))             # only shade gaps of a day or more
    print(f"{label} | {CITY['target']} | {pm.index[0]} -> {pm.index[-1]}")
    print(f"steps: {len(pm)} | missing: {int(miss.sum())} ({miss.mean():.1%}) | longest gap: {longest_gap} steps")
    print(vals.describe().round(1).to_string())

    fig = plt.figure(figsize=(16, 11))
    gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.28)

    # 1) full series + rolling mean + missing stretches
    ax = fig.add_subplot(gs[0, :])
    ax.plot(pm.index, pm.values, lw=0.3, alpha=0.5, label=CITY["target"])
    win = 30 * (24 if hourly else 1)
    ax.plot(pm.index, pm.rolling(win, min_periods=win // 4).mean(), color="C3", lw=1.5, label="30-day mean")
    ax.fill_between(pm.index, 0, 1, where=long_gap.values, transform=ax.get_xaxis_transform(),
                    color="grey", alpha=0.35, label="gap of 1+ day")
    ax.set_title(f"{label}: {CITY['target']} over time"); ax.set_ylabel(CITY["target"]); ax.legend(ncol=3, loc="upper right")

    # 2) distribution
    ax = fig.add_subplot(gs[1, 0])
    ax.hist(vals, bins=60, edgecolor="black", linewidth=0.3)
    ax.axvline(vals.mean(), color="C3", ls="--", label=f"mean {vals.mean():.0f}")
    ax.axvline(vals.median(), color="C2", ls="--", label=f"median {vals.median():.0f}")
    ax.set_title(f"Distribution (skew {vals.skew():.1f})"); ax.set_xlabel(CITY["target"]); ax.legend()

    # 3) cycle within the day (hourly data) or the week (daily data)
    ax = fig.add_subplot(gs[1, 1])
    key, xl = (pm.index.hour, "Hour of day") if hourly else (pm.index.dayofweek, "Day of week (0 = Mon)")
    g = pm.groupby(key); m = g.mean()
    ax.plot(m.index, m.values, marker="o", ms=3)
    ax.fill_between(m.index, g.quantile(0.25), g.quantile(0.75), alpha=0.25, label="25-75% range")
    ax.set_title("Average daily cycle" if hourly else "Average weekly cycle"); ax.set_xlabel(xl); ax.legend()

    # 4) seasonality across the year
    ax = fig.add_subplot(gs[1, 2])
    g = pm.groupby(pm.index.month); m = g.mean()
    ax.plot(m.index, m.values, marker="o", ms=3)
    ax.fill_between(m.index, g.quantile(0.25), g.quantile(0.75), alpha=0.25, label="25-75% range")
    ax.set_xticks(range(1, 13)); ax.set_title("Average by month of year"); ax.set_xlabel("Month"); ax.legend()

    # 5) autocorrelation (pairwise, so gaps do not break it)
    ax = fig.add_subplot(gs[2, :2])
    max_lag = 168 if hourly else 28
    acf = [pm.autocorr(l) for l in range(1, max_lag + 1)]
    ax.vlines(range(1, max_lag + 1), 0, acf, lw=1.5)
    for k in range(CITY["season"], max_lag + 1, CITY["season"]):
        ax.axvline(k, color="C3", ls=":", alpha=0.6)
    ax.set_title(f"Autocorrelation (dotted lines every {CITY['season']} steps)"); ax.set_xlabel("Lag (steps)"); ax.set_ylabel("ACF")

    # 6) how much is missing, per year
    ax = fig.add_subplot(gs[2, 2])
    by_year = miss.groupby(pm.index.year).mean() * 100
    ax.bar(by_year.index.astype(str), by_year.values, edgecolor="black")
    ax.set_title("Missing data per year (%)"); ax.set_ylabel("% of steps missing")

    plt.savefig("city_data_overview.png", dpi=150, bbox_inches="tight"); plt.show()

## 2. Windowing
Each sample: `LOOKBACK` past values -> next `H` values. Val/test windows reach back into the previous split for their inputs, so every val/test target point is forecast.

In [ ]:
def make_windows(arr, L, H):
    t = torch.tensor(arr, dtype=torch.float32)
    w = t.unfold(0, L + H, 1)                  # (num_windows, L+H)
    return w[:, :L].contiguous(), w[:, L:].contiguous()

def get_data(L, H):
    Xtr, Ytr = make_windows(z[:n_tr], L, H)
    Xva, Yva = make_windows(z[n_tr - L: n_tr + n_va], L, H)
    Xte, Yte = make_windows(z[n_tr + n_va - L:], L, H)
    return Xtr, Ytr, Xva, Yva, Xte, Yte

Xtr, Ytr, Xva, Yva, Xte, Yte = get_data(LOOKBACK, HORIZONS[0])
print("train/val/test windows:", len(Xtr), len(Xva), len(Xte))

## 3. Models

In [ ]:
class MovingAvg(nn.Module):
    def __init__(self, k):
        super().__init__(); self.k = k
    def forward(self, x):                      # x: (B, L)
        pad = (self.k - 1) // 2
        xp = torch.cat([x[:, :1].repeat(1, pad), x, x[:, -1:].repeat(1, pad)], dim=1)
        return F.avg_pool1d(xp.unsqueeze(1), self.k, stride=1).squeeze(1)

class DLinear(nn.Module):
    # Trend + seasonal decomposition, then one linear layer for each (Zeng et al. 2023)
    def __init__(self, L, H, k=25):
        super().__init__()
        self.ma = MovingAvg(k)
        self.lin_trend, self.lin_seas = nn.Linear(L, H), nn.Linear(L, H)
    def forward(self, x):
        trend = self.ma(x)
        return self.lin_seas(x - trend) + self.lin_trend(trend)

class LSTMForecaster(nn.Module):
    def __init__(self, L, H, hidden=64, layers=2, drop=0.1):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True, dropout=drop)
        self.head = nn.Linear(hidden, H)
    def forward(self, x):
        out, _ = self.lstm(x.unsqueeze(-1))
        return self.head(out[:, -1])

class TinyTransformer(nn.Module):
    # Vanilla encoder; every time step is a token (the design the paper questions)
    def __init__(self, L, H, d=64, heads=4, layers=2, ff=128, drop=0.1):
        super().__init__()
        self.inp = nn.Linear(1, d)
        self.pos = nn.Parameter(torch.randn(1, L, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Linear(L * d, H)
    def forward(self, x):
        h = self.enc(self.inp(x.unsqueeze(-1)) + self.pos)
        return self.head(h.flatten(1))

BUILDERS = {"DLinear": DLinear, "LSTM": LSTMForecaster, "Transformer": TinyTransformer}

## 4. Training and evaluation helpers

In [ ]:
@torch.no_grad()
def predict(model, X, bs=512):
    model.eval()
    return torch.cat([model(X[i:i + bs].to(device)).cpu() for i in range(0, len(X), bs)])

def mse(p, y): return ((p - y) ** 2).mean().item()
def mae(p, y): return (p - y).abs().mean().item()

def train_one(name, L, H, seed, data):
    Xtr, Ytr, Xva, Yva, Xte, Yte = data
    set_seed(seed)
    model = BUILDERS[name](L, H).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    dl = DataLoader(TensorDataset(Xtr, Ytr), batch_size=BATCH, shuffle=True)
    best, best_state, bad, t0 = float("inf"), None, 0, time.time()
    for ep in range(MAX_EPOCHS):
        model.train()
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            F.mse_loss(model(x), y).backward()
            opt.step()
        v = mse(predict(model, Xva), Yva)
        if v < best - 1e-5:
            best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    preds = predict(model, Xte)
    return {"mse": mse(preds, Yte), "mae": mae(preds, Yte), "params": n_params,
            "train_time_s": time.time() - t0, "epochs": ep + 1}, preds

def naive_preds(X, H):                         # repeat the last observed value
    return X[:, -1:].repeat(1, H)

def seasonal_naive_preds(X, H, season):        # repeat the last full season (e.g. last 24 hours)
    idx = torch.arange(H) % season
    return X[:, X.shape[1] - season + idx]

## 5. Run the main benchmark
Baselines are instant. The trained models take a few minutes in total on a T4 (3 models x 2 horizons x 3 seeds).

In [ ]:
rows, test_preds, test_truth, test_inputs = [], {}, {}, {}

for H in HORIZONS:
    data = get_data(LOOKBACK, H)
    Xte_, Yte_ = data[4], data[5]
    test_truth[H], test_inputs[H] = Yte_, Xte_

    for name, p in [("Naive (last value)", naive_preds(Xte_, H)),
                    (f"Seasonal naive ({SEASON})", seasonal_naive_preds(Xte_, H, SEASON))]:
        rows.append({"model": name, "H": H, "seed": 0, "mse": mse(p, Yte_), "mae": mae(p, Yte_),
                     "params": 0, "train_time_s": 0.0, "epochs": 0})
        test_preds[(name, H)] = p

    for name in BUILDERS:
        for seed in SEEDS:
            res, p = train_one(name, LOOKBACK, H, seed, data)
            rows.append({"model": name, "H": H, "seed": seed, **res})
            if seed == SEEDS[0]:
                test_preds[(name, H)] = p
            print(f"H={H:>3} | {name:<11} | seed {seed} | MSE {res['mse']:.4f} | MAE {res['mae']:.4f} "
                  f"| {res['params']:>9,} params | {res['train_time_s']:.0f}s | {res['epochs']} epochs")

raw = pd.DataFrame(rows)
raw.to_csv("raw_results.csv", index=False)

## 6. Results table (mean +/- std over seeds)
MSE/MAE are on the normalised scale (comparable with the paper). `MAE (orig. units)` is easier to explain.

In [ ]:
def pm(x): return f"{x.mean():.4f} +/- {x.std(ddof=0):.4f}" if len(x) > 1 else f"{x.mean():.4f}"

g = raw.groupby(["H", "model"], sort=False)
summary = g.agg(MSE=("mse", pm), MAE=("mae", pm), mse_mean=("mse", "mean"), mae_mean=("mae", "mean"),
                Params=("params", "first"), Train_s=("train_time_s", "mean")).reset_index()
summary["MAE (orig. units)"] = (summary["mae_mean"] * sd).round(3)
summary["Train_s"] = summary["Train_s"].round(1)
summary = summary.drop(columns=["mse_mean", "mae_mean"])
summary.to_csv("results_summary.csv", index=False)
for H in HORIZONS:
    print(f"\n=== Horizon {H} steps ===")
    display(summary[summary["H"] == H].drop(columns="H").reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6 * len(HORIZONS), 4.5))
axes = np.atleast_1d(axes)
for ax, H in zip(axes, HORIZONS):
    d = raw[raw["H"] == H].groupby("model", sort=False)["mse"].agg(["mean", "std"]).fillna(0)
    ax.bar(d.index, d["mean"], yerr=d["std"], capsize=4, edgecolor="black")
    ax.set_title(f"Test MSE, horizon {H}"); ax.set_ylabel("MSE (normalised)")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("mse_comparison.png", dpi=150); plt.show()

## 7. What do the forecasts look like?

In [ ]:
H = HORIZONS[-1]
i = len(test_truth[H]) // 2                      # a window in the middle of the test set
hist = test_inputs[H][i][-SEASON * 4:] * sd + mu
truth = test_truth[H][i] * sd + mu
plt.figure(figsize=(12, 4.5))
plt.plot(range(-len(hist), 0), hist, color="gray", label="History")
plt.plot(range(H), truth, color="black", lw=2, label="Truth")
for name in ["Seasonal naive (%d)" % SEASON, "DLinear", "LSTM", "Transformer"]:
    plt.plot(range(H), test_preds[(name, H)][i] * sd + mu, label=name, alpha=0.85)
plt.axvline(0, color="r", ls=":"); plt.legend(ncol=3); plt.title(f"Example forecast, horizon {H}")
plt.xlabel("Steps from forecast start"); plt.tight_layout(); plt.savefig("example_forecast.png", dpi=150); plt.show()

## 8. (Optional, ~5-10 min) Does a longer look-back help?
The paper's headline finding: DLinear keeps improving with a longer look-back window, while Transformers often don't. This runs one seed per setting.

In [ ]:
H_SWEEP, LOOKBACKS = HORIZONS[0], [48, 96, 168, 336, 504]
sweep = []
for L in LOOKBACKS:
    if L + H_SWEEP > n_tr:
        continue
    data = get_data(L, H_SWEEP)
    for name in ["DLinear", "Transformer"]:
        res, _ = train_one(name, L, H_SWEEP, 0, data)
        sweep.append({"model": name, "lookback": L, "mse": res["mse"]})
        print(f"L={L:>3} | {name:<11} | MSE {res['mse']:.4f}")
df_sw = pd.DataFrame(sweep)
df_sw.to_csv("lookback_sweep.csv", index=False)
plt.figure(figsize=(6, 4.5))
for name, d in df_sw.groupby("model"):
    plt.plot(d["lookback"], d["mse"], marker="o", label=name)
plt.xlabel("Look-back window (steps)"); plt.ylabel("Test MSE"); plt.title(f"Look-back sweep, horizon {H_SWEEP}")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig("lookback_sweep.png", dpi=150); plt.show()

## 9. Download your results for the report/README

In [ ]:
from google.colab import files
for f in ["results_summary.csv", "raw_results.csv", "mse_comparison.png", "example_forecast.png",
          "data_overview.png", "lookback_sweep.png", "lookback_sweep.csv", "city_data_overview.png"]:
    if os.path.exists(f):
        files.download(f)